# The Bimodal Biometric Warehouse
## Mining Wearable and Nutritional Data for Metabolic Optimization

| Phase | Topic | Key Technique |
|-------|-------|---------------|
| Part 1 | Data Engineering & Star-Schema Warehouse | Surrogate keys, `pd.merge()` on `full_date` |
| Part 2 | Preprocessing, EDA & Visualisation | PCA on 365 × 1440 minute-level HR matrix |
| Part 3 | Data Mining Engine | K-Means, Apriori, Regression, RF + SVM with lag features |
| Part 4 | Evaluation & Deployment | sklearn `Pipeline` (scaler+model) pickled together |


## 0  Environment Setup

In [ ]:
import os, sys, warnings, sqlite3, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

SEED = 42
np.random.seed(SEED)

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH  = os.path.join(BASE_DIR, "database", "biometric_warehouse.db")
DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR  = os.path.join(BASE_DIR, "outputs")
os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})
sns.set_theme(style="whitegrid")
print("✓ Setup complete")


---
## Part 1 — Data Engineering & Warehousing

### 1.1  Star Schema (Surrogate Keys)

```
                ┌─────────────────────────┐
                │        Dim_Date          │
                │  date_sk PK (surrogate)  │
                │  full_date  (natural key) │
                └────────────┬────────────┘
                             │
┌──────────────────┐  ┌──────┴────────────────────────────┐  ┌──────────────────┐
│   Dim_Workout    │  │     Fact_Daily_Biometrics          │  │  Dim_Nutrition   │
│ workout_sk  PK   │──│  fact_sk  PK  (surrogate)         │──│ nutrition_sk  PK │
│ full_date   (NK) │  │  date_sk  FK → Dim_Date            │  │ full_date    (NK)│
│ workout_type     │  │  workout_sk  FK → Dim_Workout      │  │ meal_category    │
│ intensity        │  │  nutrition_sk FK → Dim_Nutrition   │  │ protein_g        │
│ duration_minutes │  │  hr_pc1..hr_pc5 (from PCA)        │  │ is_high_protein  │
└──────────────────┘  │  lag1_sleep / lag1_active_cal      │  └──────────────────┘
                      │  recovery_score / recovery_label   │
                      └───────────────────────────────────┘
```

### 1.2  Build raw sources & load warehouse


In [ ]:
import subprocess
subprocess.run([sys.executable,
                os.path.join(BASE_DIR, "data", "generate_data.py")], check=True)
subprocess.run([sys.executable,
                os.path.join(BASE_DIR, "database", "load_data.py")], check=True)


In [ ]:
# ── 1.3 Load dimension + fact tables via SQLAlchemy ──────────────────────────
engine = create_engine(f"sqlite:///{DB_PATH}")

dim_date      = pd.read_sql("SELECT * FROM Dim_Date",             engine)
dim_workout   = pd.read_sql("SELECT * FROM Dim_Workout",          engine)
dim_nutrition = pd.read_sql("SELECT * FROM Dim_Nutrition",        engine)
fact_raw      = pd.read_sql("SELECT * FROM Fact_Daily_Biometrics",engine)

for name, tbl in [("Dim_Date", dim_date), ("Dim_Workout", dim_workout),
                   ("Dim_Nutrition", dim_nutrition),
                   ("Fact_Daily_Biometrics", fact_raw)]:
    print(f"  {name:<24} {tbl.shape[0]} rows × {tbl.shape[1]} cols")


In [ ]:
# ── 1.4 Analytical frame: full star-schema JOIN (SQL) ─────────────────────────
query = """
SELECT
    f.fact_sk, f.recovery_score, f.recovery_label,
    f.total_active_minutes, f.resting_heart_rate, f.sleep_duration_hours,
    f.active_calories, f.steps, f.hrv_score,
    dd.full_date, dd.day_of_week, dd.month, dd.quarter, dd.season, dd.is_weekend,
    dw.workout_type, dw.exercise_category, dw.intensity, dw.duration_minutes,
    dn.meal_category, dn.total_calories, dn.protein_g, dn.carbs_g, dn.fat_g,
    dn.is_high_protein, dn.is_poultry, dn.is_vegetarian
FROM Fact_Daily_Biometrics f
JOIN Dim_Date      dd ON f.date_sk      = dd.date_sk
JOIN Dim_Workout   dw ON f.workout_sk   = dw.workout_sk
JOIN Dim_Nutrition dn ON f.nutrition_sk = dn.nutrition_sk
ORDER BY dd.full_date
"""
df = pd.read_sql(query, engine)
df["full_date"] = pd.to_datetime(df["full_date"])
print(f"Analytical frame: {df.shape[0]} rows × {df.shape[1]} cols")
df.head(3)


---
## Part 2 — Preprocessing, EDA & Visualisation

### 2.1  Missing-value audit & mean imputation

The wearable sensor occasionally drops readings (~2 % of minutes).
We detect those gaps and apply **mean imputation** before PCA.


In [ ]:
# Simulate sensor dropouts then impute
rng_local = np.random.default_rng(SEED)
mask_rhr = rng_local.random(len(df)) < 0.02
mask_hrv = rng_local.random(len(df)) < 0.02
df.loc[mask_rhr, "resting_heart_rate"] = np.nan
df.loc[mask_hrv, "hrv_score"]          = np.nan

print(f"Simulated gaps — resting_heart_rate: {mask_rhr.sum()}, hrv_score: {mask_hrv.sum()}")
df["resting_heart_rate"] = df["resting_heart_rate"].fillna(df["resting_heart_rate"].mean())
df["hrv_score"]          = df["hrv_score"].fillna(df["hrv_score"].mean())
print("Mean imputation applied ✓")


### 2.2  PCA on the 365 × 1440 Minute-Level Heart-Rate Matrix

> **Project differentiator**: instead of summarising HR to a single daily mean,
> we pivot the raw wearable export into a matrix where every row is a calendar
> day and every column is a minute of the day (0 – 1439).
> PCA on this 365 × 1440 matrix extracts *waveform* features.
>
> * **PC1** — overall daily exertion (high loadings at workout minutes)
> * **PC2** — bimodal vs unimodal signature (differentiates a morning workout +
>   afternoon walk from a flat sedentary day)
> * **PC3+** — timing shifts (early vs late workout)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ── load & pivot minute-level wearable data ───────────────────────────────────
print("Loading minute-level HR data …")
wearable_raw = pd.read_csv(os.path.join(DATA_DIR, "wearable_raw.csv"))

# Pivot: rows = dates, columns = minutes 0–1439
hr_matrix = (
    wearable_raw
    .pivot(index="full_date", columns="minute", values="heart_rate")
    .sort_index()
)
print(f"HR matrix shape: {hr_matrix.shape}  (days × minutes)")

# Fill any rare NaN pivots with row mean (sensor dropout)
hr_matrix = hr_matrix.apply(lambda row: row.fillna(row.mean()), axis=1)


In [ ]:
# ── Fit PCA on the 1440-dim HR waveforms ─────────────────────────────────────
scaler_hr = StandardScaler()
H = scaler_hr.fit_transform(hr_matrix.values)   # (365, 1440), zero-mean unit-var

pca_hr = PCA(n_components=5, random_state=SEED)
hr_pcs = pca_hr.fit_transform(H)                # (365, 5)

# Attach to analytical dataframe (align on date)
pc_df = pd.DataFrame(hr_pcs,
                     index=hr_matrix.index,
                     columns=["hr_pc1","hr_pc2","hr_pc3","hr_pc4","hr_pc5"])
pc_df.index = pd.to_datetime(pc_df.index)
df = df.merge(pc_df, left_on="full_date", right_index=True, how="left")

evr = pca_hr.explained_variance_ratio_
print("PCA on 1440-dim HR waveforms — Explained Variance:")
for i, v in enumerate(evr, 1):
    print(f"  PC{i}: {v:.1%}")
print(f"  Cumulative (5 PCs): {evr.sum():.1%}")


In [ ]:
# ── Visualise average waveform per cluster (exercise category) ────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
palette = {"Cardio":"#e74c3c", "Strength":"#3498db",
           "Rest":"#95a5a6", "Recovery":"#2ecc71"}

for ax, cat in zip(axes, ["Cardio","Strength","Rest"]):
    dates_cat = (
        df[df["exercise_category"] == cat]["full_date"]
        .dt.strftime("%Y-%m-%d").values
    )
    subset = hr_matrix.loc[hr_matrix.index.isin(dates_cat)]
    mean_wave = subset.mean(axis=0).values
    ax.plot(range(1440), mean_wave, color=palette[cat], lw=1.5)
    ax.axhline(mean_wave.min()+5, color="gray", linestyle="--", lw=0.8)
    ax.set_title(f"Average HR Waveform — {cat} days (n={len(subset)})")
    ax.set_xlabel("Minute of Day"); ax.set_ylabel("Heart Rate (bpm)")
    ax.set_xticks([0, 360, 720, 1080, 1440])
    ax.set_xticklabels(["12am","6am","12pm","6pm","12am"])

plt.suptitle("Bimodal HR Signature: Two Peaks on Cardio/Strength Days", y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "bimodal_hr_waveforms.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ── PCA component loadings (which minutes drive each PC) ─────────────────────
fig, axes = plt.subplots(5, 1, figsize=(14, 14), sharex=True)
for i, ax in enumerate(axes):
    loading = pca_hr.components_[i]  # shape (1440,)
    ax.plot(range(1440), loading, lw=0.8, color=f"C{i}")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_ylabel(f"PC{i+1} loading\n({evr[i]:.1%})")
    ax.set_xlim(0, 1440)
    if i == 4:
        ax.set_xticks([0, 360, 720, 1080, 1440])
        ax.set_xticklabels(["12am","6am","12pm","6pm","12am"])
        ax.set_xlabel("Minute of Day")

plt.suptitle("PCA Component Loadings on 1440-Minute HR Waveform\n"
             "(PC1=Total Exertion · PC2=Bimodal vs Flat · PC3–5=Timing)", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "pca_loadings_1440.png"), bbox_inches="tight")
plt.show()


In [ ]:
# ── 2.3 Lag features — yesterday's sleep, calories, HRV ──────────────────────
df = df.sort_values("full_date").reset_index(drop=True)
df["lag1_sleep"]           = df["sleep_duration_hours"].shift(1)
df["lag1_active_calories"] = df["active_calories"].shift(1)
df["lag1_hrv"]             = df["hrv_score"].shift(1)

# Drop day 0 (no lag available) then reset
df = df.dropna(subset=["lag1_sleep","lag1_active_calories","lag1_hrv"]).reset_index(drop=True)
print(f"Analytical frame after lag: {df.shape} (dropped 1 row for lag)")


In [ ]:
# ── 2.4 Histograms ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
metrics = [
    ("resting_heart_rate",   "Resting HR (bpm)"),
    ("hrv_score",            "HRV Score"),
    ("sleep_duration_hours", "Sleep Duration (h)"),
    ("active_calories",      "Active Calories"),
    ("steps",                "Daily Steps"),
    ("protein_g",            "Protein Intake (g)"),
]
for ax, (col, lbl) in zip(axes.flat, metrics):
    ax.hist(df[col], bins=25, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.set_xlabel(lbl); ax.set_ylabel("Count")
    ax.set_title(f"Distribution: {lbl}")
plt.suptitle("Biometric & Nutritional Distributions (365 Days)", y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "histograms.png"), bbox_inches="tight"); plt.show()


In [ ]:
# ── 2.5 Correlation heatmap ───────────────────────────────────────────────────
numeric_cols = [
    "resting_heart_rate","hrv_score","sleep_duration_hours","active_calories",
    "steps","total_active_minutes","protein_g","carbs_g","fat_g",
    "recovery_score","intensity","duration_minutes",
    "hr_pc1","hr_pc2","hr_pc3",
]
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, linewidths=0.4, ax=ax, vmin=-1, vmax=1, annot_kws={"size":7})
ax.set_title("Feature Correlation Matrix (includes HR PCA components)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "correlation_heatmap.png")); plt.show()


In [ ]:
# ── 2.6 High-protein days vs. next-day resting HR ────────────────────────────
df["next_day_rhr"] = df["resting_heart_rate"].shift(-1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df.boxplot(column="next_day_rhr", by="is_high_protein", ax=axes[0])
axes[0].set_xticklabels(["Low Protein (≤150 g)","High Protein (>150 g)"])
axes[0].set_title("Next-Day RHR by Protein Intake")
axes[0].set_ylabel("Next-Day RHR (bpm)"); axes[0].set_xlabel("")
plt.sca(axes[0]); plt.suptitle("")

from matplotlib.patches import Patch
colors_pt = df["is_high_protein"].map({0:"#E07B54",1:"#4C72B0"})
axes[1].scatter(df["protein_g"],
                df["next_day_rhr"].fillna(df["next_day_rhr"].mean()),
                c=colors_pt, alpha=0.55, s=25, edgecolors="none")
m, b = np.polyfit(df["protein_g"],
                  df["next_day_rhr"].fillna(df["next_day_rhr"].mean()), 1)
xl = np.linspace(df["protein_g"].min(), df["protein_g"].max(), 200)
axes[1].plot(xl, m*xl+b, color="black", lw=1.5, label=f"Trend (slope={m:.3f})")
axes[1].set_xlabel("Protein (g)"); axes[1].set_ylabel("Next-Day RHR (bpm)")
axes[1].set_title("Protein Intake → Next-Day Resting HR")
handles = [Patch(color="#E07B54",label="Low Protein"),
           Patch(color="#4C72B0",label="High Protein")]
axes[1].legend(handles=handles)

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "protein_vs_rhr.png")); plt.show()

for label, grp in df.groupby("is_high_protein"):
    lbl = "High protein" if label else "Low protein"
    print(f"Mean next-day RHR | {lbl:<14}: {grp['next_day_rhr'].mean():.1f} bpm")


---
## Part 3 — The Data Mining Engine

### 3.1 Unsupervised: K-Means Day Clustering
Uses HR PCA components + biometric features so the cluster shape reflects
the minute-level waveform, not just daily totals.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

cluster_features = [
    "hr_pc1","hr_pc2","hr_pc3",               # waveform shape
    "total_active_minutes","resting_heart_rate",
    "sleep_duration_hours","hrv_score","intensity","protein_g",
]
X_clust = df[cluster_features].fillna(df[cluster_features].mean()).copy()
scaler_k = StandardScaler()
X_cs = scaler_k.fit_transform(X_clust)

inertias = []
for k in range(2, 10):
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    km.fit(X_cs)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(2, 10), inertias, "o-", color="#4C72B0", lw=2)
ax.axvline(3, color="red", linestyle="--", lw=1, label="Chosen k=3")
ax.set_xlabel("k"); ax.set_ylabel("Inertia")
ax.set_title("K-Means Elbow Curve (HR PCA + biometric features)")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "kmeans_elbow.png")); plt.show()


In [ ]:
km3 = KMeans(n_clusters=3, random_state=SEED, n_init=10)
df["cluster"] = km3.fit_predict(X_cs)

# Name clusters by mean recovery score
order = df.groupby("cluster")["recovery_score"].mean().sort_values(ascending=False).index
label_map = {order[0]:"Optimal Balance",
             order[1]:"High Strain / Low Recovery",
             order[2]:"Sedentary"}
df["cluster_name"] = df["cluster"].map(label_map)

profile_cols = ["hr_pc1","hr_pc2","total_active_minutes",
                "resting_heart_rate","sleep_duration_hours",
                "hrv_score","intensity","protein_g","recovery_score"]
print("Cluster Profiles:")
print(df.groupby("cluster_name")[profile_cols].mean().round(1).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
colors_c = {"Optimal Balance":"#2ecc71",
            "High Strain / Low Recovery":"#e74c3c",
            "Sedentary":"#3498db"}
for name, grp in df.groupby("cluster_name"):
    ax.scatter(grp["hr_pc1"], grp["hr_pc2"],
               label=name, color=colors_c[name], alpha=0.65, s=30, edgecolors="none")
ax.set_xlabel("HR-PC1 (Overall Exertion)")
ax.set_ylabel("HR-PC2 (Bimodal vs Flat Signature)")
ax.set_title("K-Means Clusters projected onto HR Waveform PCA Space")
ax.legend(); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "kmeans_clusters.png")); plt.show()


### 3.2 Association Rule Mining (Apriori)

Every feature is binarised into a boolean transaction item before applying Apriori.


In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

def day_to_basket(row):
    """Convert one row of the analytical frame into a list of boolean items."""
    items = []
    # Meal
    items.append(f"Meal={row['meal_category']}")
    if row["is_high_protein"]:  items.append("HighProtein")
    if row["is_poultry"]:       items.append("Poultry")
    if row["is_vegetarian"]:    items.append("Vegetarian")
    # Workout
    items.append(f"Workout={row['workout_type']}")
    items.append(f"Category={row['exercise_category']}")
    # Sleep quality
    items.append("Sleep=High" if row["sleep_duration_hours"] >= 7.5 else "Sleep=Low")
    # Recovery
    if   row["recovery_score"] >= 70: items.append("Recovery=Excellent")
    elif row["recovery_score"] >= 55: items.append("Recovery=Good")
    else:                              items.append("Recovery=Poor")
    # HRV tier
    items.append("HRV=High"   if row["hrv_score"] >= 60 else "HRV=Low")
    # HR-PC1 tier (overall exertion captured from waveform)
    items.append("Exertion=High" if row["hr_pc1"] > 0 else "Exertion=Low")
    # Lag: did yesterday have high calories?
    if row["lag1_active_calories"] > df["lag1_active_calories"].median():
        items.append("YesterdayHighCal")
    # Lag: was yesterday's sleep long?
    if row["lag1_sleep"] >= 7.5:
        items.append("YesterdayGoodSleep")
    return items

transactions = df.apply(day_to_basket, axis=1).tolist()
te = TransactionEncoder()
basket_df = pd.DataFrame(te.fit_transform(transactions), columns=te.columns_)

freq_items = apriori(basket_df, min_support=0.05, use_colnames=True)
rules = association_rules(freq_items, metric="lift", min_threshold=1.1)
rules = rules.sort_values("lift", ascending=False)
print(f"Frequent itemsets: {len(freq_items)} | Association rules: {len(rules)}")


In [ ]:
display_cols = ["antecedents","consequents","support","confidence","lift"]
top = rules[display_cols].head(15).copy()
top["antecedents"] = top["antecedents"].apply(lambda x: ", ".join(sorted(x)))
top["consequents"] = top["consequents"].apply(lambda x: ", ".join(sorted(x)))
top.rename(columns={"antecedents":"IF","consequents":"THEN"}, inplace=True)
top[["support","confidence","lift"]] = top[["support","confidence","lift"]].round(3)
print("\nTop 15 Rules (sorted by Lift):")
print(top.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sc = ax.scatter(rules["support"], rules["confidence"],
                c=rules["lift"], cmap="YlOrRd", alpha=0.7, s=40, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Lift")
ax.set_xlabel("Support"); ax.set_ylabel("Confidence")
ax.set_title("Apriori Association Rules — Support vs Confidence (colour = Lift)")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "apriori_rules.png")); plt.show()


### 3.3 Regression — Predicting Active Calorie Burn

Multi-variable linear regression using protein intake, workout intensity, and
HR waveform PCA components as predictors.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline

reg_features = [
    "protein_g","intensity","duration_minutes",
    "steps","total_active_minutes","carbs_g",
    "hr_pc1","hr_pc2",                         # waveform features
]
X_reg = df[reg_features].fillna(df[reg_features].mean())
y_reg = df["active_calories"]

X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, test_size=0.2, random_state=SEED)

# ── Bundle scaler + regressor into a Pipeline ─────────────────────────────────
reg_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("lr",     LinearRegression()),
])
reg_pipeline.fit(X_tr, y_tr)
y_pred_reg = reg_pipeline.predict(X_te)

rmse = root_mean_squared_error(y_te, y_pred_reg)
mae  = mean_absolute_error(y_te, y_pred_reg)
r2   = r2_score(y_te, y_pred_reg)
print(f"Regression Pipeline — Active Calorie Prediction")
print(f"  RMSE : {rmse:.2f} kcal")
print(f"  MAE  : {mae:.2f} kcal")
print(f"  R²   : {r2:.4f}")

lr_step = reg_pipeline.named_steps["lr"]
print("\nCoefficients (unscaled space approximation):")
for feat, coef in sorted(zip(reg_features, lr_step.coef_),
                          key=lambda x: abs(x[1]), reverse=True):
    print(f"  {feat:<28} {coef:+.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_te, y_pred_reg, alpha=0.6, s=25, color="#4C72B0", edgecolors="none")
mn, mx = min(y_te.min(), y_pred_reg.min()), max(y_te.max(), y_pred_reg.max())
axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect fit")
axes[0].set_xlabel("Actual Active Calories"); axes[0].set_ylabel("Predicted")
axes[0].set_title(f"Regression: Actual vs Predicted  R²={r2:.3f}")
axes[0].legend()

residuals = y_te - y_pred_reg
axes[1].hist(residuals, bins=25, color="#55A868", edgecolor="white")
axes[1].axvline(0, color="red", linestyle="--", lw=1.5)
axes[1].set_xlabel("Residual"); axes[1].set_ylabel("Count")
axes[1].set_title(f"Residual Distribution  RMSE={rmse:.1f} kcal")

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "regression_results.png")); plt.show()


### 3.4 Classification — Predicting Tomorrow's Recovery State

Feature set includes:
- HR waveform PCA components (hr_pc1–hr_pc5)
- **Lag features** — yesterday's sleep, active calories, and HRV
- All biometric + nutrition flags


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

clf_features = [
    "resting_heart_rate","hrv_score","sleep_duration_hours",
    "total_active_minutes","active_calories","steps",
    "protein_g","carbs_g","fat_g",
    "intensity","duration_minutes",
    "is_high_protein","is_poultry","is_vegetarian","is_weekend",
    "hr_pc1","hr_pc2","hr_pc3","hr_pc4","hr_pc5",   # waveform PCA
    "lag1_sleep","lag1_active_calories","lag1_hrv",  # lag features
]
X_clf = df[clf_features].fillna(df[clf_features].mean())
y_clf = df["recovery_label"]

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=SEED, stratify=y_clf
)

# ── Pipeline: StandardScaler + RandomForest ───────────────────────────────────
rf_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(n_estimators=200, max_depth=8,
                                       class_weight="balanced", random_state=SEED)),
])
rf_pipeline.fit(X_tr_c, y_tr_c)
y_pred_rf   = rf_pipeline.predict(X_te_c)
y_proba_rf  = rf_pipeline.predict_proba(X_te_c)[:, 1]

# ── Pipeline: StandardScaler + SVM ───────────────────────────────────────────
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svc",    SVC(kernel="rbf", C=5, gamma="scale",
                   probability=True, class_weight="balanced", random_state=SEED)),
])
svm_pipeline.fit(X_tr_c, y_tr_c)
y_pred_svm  = svm_pipeline.predict(X_te_c)
y_proba_svm = svm_pipeline.predict_proba(X_te_c)[:, 1]

print("Pipelines trained: (StandardScaler + RF) and (StandardScaler + SVM) ✓")


---
## Part 4 — Evaluation & Deployment

### 4.1 Classification Evaluation: Confusion Matrix · F1 · ROC-AUC


In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report, f1_score,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

for name, y_pred, y_proba in [
    ("Random Forest Pipeline", y_pred_rf,  y_proba_rf),
    ("SVM Pipeline (RBF)",     y_pred_svm, y_proba_svm),
]:
    f1  = f1_score(y_te_c, y_pred)
    auc = roc_auc_score(y_te_c, y_proba)
    print(f"\n{'─'*45}")
    print(f"  {name}")
    print(f"{'─'*45}")
    print(f"  F1 Score : {f1:.4f}")
    print(f"  ROC-AUC  : {auc:.4f}")
    print(classification_report(y_te_c, y_pred,
                                 target_names=["Needs Rest","Ready to Train"]))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, y_pred) in zip(axes, [
    ("Random Forest Pipeline", y_pred_rf),
    ("SVM Pipeline (RBF)",     y_pred_svm),
]):
    ConfusionMatrixDisplay(
        confusion_matrix(y_te_c, y_pred),
        display_labels=["Needs Rest","Ready to Train"]
    ).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"Confusion Matrix — {name}")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrices.png")); plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, y_proba in [("RF Pipeline",  y_proba_rf),
                       ("SVM Pipeline", y_proba_svm)]:
    fpr, tpr, _ = roc_curve(y_te_c, y_proba)
    auc = roc_auc_score(y_te_c, y_proba)
    ax.plot(fpr, tpr, lw=2, label=f"{name}  AUC={auc:.3f}")
ax.plot([0,1],[0,1],"k--",lw=1)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Recovery State Classification")
ax.legend(loc="lower right"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "roc_curves.png")); plt.show()


### 4.2 Feature Importance (Random Forest)

In [ ]:
rf_step = rf_pipeline.named_steps["clf"]
feat_imp = (pd.Series(rf_step.feature_importances_, index=clf_features)
             .sort_values(ascending=True))

fig, ax = plt.subplots(figsize=(9, 9))
feat_imp.plot(kind="barh", ax=ax, color="#4C72B0", edgecolor="white")
ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Random Forest — Feature Importance\n(lag features + HR waveform PCA components highlighted)")
# Highlight lag + PCA features
highlight = {f for f in clf_features
             if f.startswith("lag") or f.startswith("hr_pc")}
for label in ax.get_yticklabels():
    if label.get_text() in highlight:
        label.set_color("#e74c3c")
        label.set_fontweight("bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "feature_importance.png")); plt.show()
print("Red labels = lag features and HR PCA waveform components")


### 4.3 Regression Metrics

| Metric | Value |
|--------|-------|
| RMSE | — |
| MAE  | — |
| R²   | — |

*(Values filled in at runtime above)*


In [ ]:
print("Linear Regression Pipeline — Active Calorie Expenditure")
print(f"  RMSE : {rmse:.2f} kcal")
print(f"  MAE  : {mae:.2f} kcal")
print(f"  R²   : {r2:.4f}")


### 4.4 Pickle the Full Pipelines (Scaler + Model Together)

> **Critical detail**: the `StandardScaler` is inside the `Pipeline` object,
> so a single `joblib.dump(rf_pipeline, ...)` serialises both the fitted scaler
> AND the fitted Random Forest.  The Flask API just calls `pipeline.predict()`
> on raw input — no manual scaling required and no risk of data leakage.


In [ ]:
import joblib

models_dir = os.path.join(BASE_DIR, "app", "models")
os.makedirs(models_dir, exist_ok=True)

# The entire Pipeline (scaler + model) is pickled — not just the model
joblib.dump(rf_pipeline,  os.path.join(models_dir, "rf_pipeline.pkl"))
joblib.dump(svm_pipeline, os.path.join(models_dir, "svm_pipeline.pkl"))
joblib.dump(reg_pipeline, os.path.join(models_dir, "reg_pipeline.pkl"))

# Also pickle the 1440-dim HR PCA pipeline so the API can compute PC scores
hr_pca_pipeline = Pipeline([
    ("scaler", scaler_hr),
    ("pca",    pca_hr),
])
joblib.dump(hr_pca_pipeline, os.path.join(models_dir, "hr_pca_pipeline.pkl"))

# Feature name lists for API validation
with open(os.path.join(models_dir, "clf_features.json"), "w") as fh:
    json.dump(clf_features, fh)
with open(os.path.join(models_dir, "reg_features.json"), "w") as fh:
    json.dump(reg_features, fh)

print("✓ Saved pipelines:")
for fname in os.listdir(models_dir):
    size = os.path.getsize(os.path.join(models_dir, fname))
    print(f"  {fname:<30} {size/1024:.1f} KB")


### 4.5 Flask API — Live Prediction

> Run with:
> ```bash
> python app/flask_api.py
> ```
>
> The API loads the full `Pipeline` objects.  Input is raw (unscaled) JSON — 
> scaling happens automatically inside `pipeline.predict()`.
>
> ```bash
> curl -X POST http://localhost:5000/predict \\
>   -H "Content-Type: application/json" \\
>   -d '{"protein_g":180,"intensity":7,"duration_minutes":50, ...}'
> ```
>
> Response:
> ```json
> {
>   "recovery_label": 1,
>   "recovery_probability": 0.83,
>   "message": "Ready to Train",
>   "predicted_active_calories": 612.4
> }
> ```


---
## Summary

| Step | Technique | Implementation Detail |
|------|-----------|----------------------|
| Data Engineering | SQLite star schema | Surrogate keys (date_sk, workout_sk, nutrition_sk); `pd.merge()` on `full_date` |
| PCA on HR | 365 × 1440 minute-level matrix | PC1 = exertion, PC2 = bimodal vs flat, 5 PCs pickled in pipeline |
| Preprocessing | Mean imputation + lag features | lag1_sleep, lag1_active_calories, lag1_hrv |
| K-Means | k=3 on HR-PC + biometrics | Clusters: Optimal / High Strain / Sedentary |
| Apriori | Full binarisation of all features | Lag flags (YesterdayHighCal) included in transactions |
| Regression | `Pipeline([scaler, lr])` | RMSE & R² on active calorie prediction |
| Classification | `Pipeline([scaler, rf])` + `Pipeline([scaler, svm])` | F1, ROC-AUC on recovery label |
| Deployment | Flask REST API | Loads full Pipeline — scaler always applied automatically |
